# SVM 实验 — 支持向量机分类与回归

**目标数据集：** [Iris (鸢尾花)](https://scikit-learn.org/stable/datasets/toy_dataset.html#iris-dataset) / [California Housing](https://scikit-learn.org/stable/datasets/real_world.html#california-housing-dataset)

**核心任务：** 使用支持向量机 (SVM) 进行分类与回归，掌握核函数选择、网格搜索调参、特征选择与提取对模型性能的影响

**实验流程：**

| 步骤 | 内容 | 掌握技能 |
|------|------|----------|
| 1 | 环境验证 | Python / scikit-learn / matplotlib / mlxtend |
| 2 | SVM 简介 | 原理 / 超平面 / 支持向量 / 核函数 |
| 3 | scikit-learn 简介 | API 设计 / 模型接口 / 交叉验证 |
| 4 | SVM 分类 + 交叉验证 | SVC / cross_val_score / 决策边界可视化 |
| 5 | 特征选择与提取对比 | Pipeline / SelectKBest / PCA |
| 6 | 核函数与网格搜索 | GridSearchCV / kernel / C / gamma |
| 7 | SVR 回归 | 加州房价 / SVR / 回归评估指标 |
| 8 | SVR 探索* | 特征选择 / PCA / RandomizedSearchCV |

> 注：标星号(*)的节为进阶探索内容。

In [ ]:
import sys
print(f"Python: {sys.version}")

import numpy as np
print(f"numpy: {np.__version__}")

import sklearn
print(f"scikit-learn: {sklearn.__version__}")

import matplotlib
print(f"matplotlib: {matplotlib.__version__}")

import pandas as pd
print(f"pandas: {pd.__version__}")

import mlxtend
print(f"mlxtend: {mlxtend.__version__}")

import seaborn as sns
print(f"seaborn: {sns.__version__}")

print("\n环境就绪 ✓")

---
## 2. SVM 简介

支持向量机 (Support Vector Machine, SVM) 是一种强大的监督学习算法，可用于分类和回归任务。其核心思想是找到最优的决策边界（超平面），使不同类别的样本尽可能分开。

### 2.1 基本概念

**超平面（Hyperplane）**：在 d 维空间中，超平面是一个 d-1 维的子空间，用于划分不同类别的数据点。

**支持向量（Support Vectors）**：距离超平面最近的样本点，它们决定了超平面的位置。

**间隔（Margin）**：超平面到最近支持向量的距离。SVM 的目标是最大化间隔。

```
超平面与支持向量示意
────────────────────

      类别 A          类别 B
    ○ ○ ○           ● ● ●
  ○ ○ ⊙ ○ ○       ● ● ⊙ ● ●
○ ○ ○ ○ ○ ○ ○   ● ● ● ● ● ● ●
    ────────────           ← 超平面 (决策边界)
        ║                   ║
        ║ ← 间隔 → ║
        ║                   ║
      ⊙                   ⊙    ← 支持向量
```

### 2.2 核函数

| 核函数 | 公式 | 适用场景 |
|--------|------|----------|
| linear | $K(x, x') = x \cdot x'$ | 线性可分数据 |
| rbf (Gaussian) | $K(x, x') = \exp(-\gamma \|x - x'\|^2)$ | 非线性数据，最常用 |
| poly | $K(x, x') = (x \cdot x' + c)^d$ | 特定结构数据 |

### 2.3 关键超参数

| 参数 | 含义 | 影响 |
|------|------|------|
| C | 惩罚系数 | C 越大，对误分类容忍度越低 |
| gamma | RBF 核的宽度 | gamma 越大，单个样本影响范围越小 |
| kernel | 核函数类型 | 决定数据的映射方式 |

---
## 3. scikit-learn 简介

scikit-learn 是 Python 最流行的机器学习库，提供了统一的 API 接口。

### 3.1 核心 API 设计

| 方法 | 功能 |
|------|------|
| `fit(X, y)` | 训练模型 |
| `predict(X)` | 对新数据进行预测 |
| `score(X, y)` | 评估模型性能 |

### 3.2 SVM 相关类

| 类名 | 任务 | 说明 |
|------|------|------|
| `SVC` | 分类 | Support Vector Classification |
| `SVR` | 回归 | Support Vector Regression |

---
## 4. SVM 分类 + 交叉验证

本节使用 SVC 对 Iris 数据集进行分类。

### 4.1 数据加载与可视化

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn import datasets
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.svm import SVC
from sklearn.metrics import classification_report
from mlxtend.plotting import plot_decision_regions

# 定义配色方案（与 02/03 保持一致）
COLORS = ["#e74c3c", "#2ecc71", "#3498db"]
LABELS = ['setosa', 'versicolor', 'virginica']

# 加载数据
iris = datasets.load_iris()
X = iris.data[:, [0, 2]]  # 选择萼片长度和花瓣长度两个特征
y = iris.target

print(f"样本数: {len(X)}, 特征数: {X.shape[1]}")
print(f"类别: {LABELS}")

# 绘制散点图
plt.figure(figsize=(10, 6))
for i, name in enumerate(LABELS):
    mask = y == i
    plt.scatter(X[mask, 0], X[mask, 1], c=COLORS[i], label=name, 
                edgecolors="w", s=50, alpha=0.8)
plt.xlabel('Sepal Length (cm)')
plt.ylabel('Petal Length (cm)')
plt.title('Iris Dataset - Feature Visualization')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### 4.2 数据划分

使用 `train_test_split` 将数据划分为训练集（70%）和测试集（30%）。

| 参数 | 说明 |
|------|------|
| `test_size=0.3` | 测试集占总数据的 30% |
| `random_state=42` | 固定随机种子，保证结果可复现 |

In [ ]:
# 划分数据集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"训练集样本数: {len(X_train)}")
print(f"测试集样本数: {len(X_test)}")

### 4.3 交叉验证与模型训练

**交叉验证（Cross Validation）**：将数据分成 k 个子集（fold），轮流使用其中 k-1 个作为训练集，1 个作为验证集。

| 参数 | 说明 |
|------|------|
| `cv=5` | 5-fold 交叉验证 |
| `scoring='accuracy'` | 使用准确率作为评估指标 |
| `kernel='linear'` | 使用线性核函数 |
| `C=1` | 惩罚系数为 1 |

In [ ]:
# 创建模型并进行交叉验证
model = SVC(kernel='linear', C=1)
scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')

print(f"交叉验证准确率: {scores.mean():.4f} ± {scores.std():.4f}")
print(f"各折得分: {scores}")

### 4.4 决策边界可视化

训练模型并可视化决策边界，展示模型如何在特征空间中划分不同类别的区域。

In [ ]:
# 训练模型并可视化决策边界
model.fit(X_train, y_train)

plt.figure(figsize=(10, 6))
plot_decision_regions(X_train, y_train, clf=model, legend=2)
plt.xlabel('Sepal Length (cm)')
plt.ylabel('Petal Length (cm)')
plt.title('SVM Decision Boundary (Linear Kernel)')
plt.show()

### 4.5 分类报告

使用 `classification_report` 输出详细的分类性能指标：

| 指标 | 含义 |
|------|------|
| precision | 精确率：预测为正的样本中实际为正的比例 |
| recall | 召回率：实际为正的样本中被正确预测的比例 |
| f1-score | F1 值：精确率和召回率的调和平均 |
| support | 该类别的样本数 |

In [ ]:
# 在测试集上进行预测
y_pred = model.predict(X_test)
# 生成分类报告
report = classification_report(
    y_true=y_test,
    y_pred=y_pred,
    target_names=iris.target_names  # 显示类别名称
)
# 打印报告
print("分类性能报告:")
print(report)

---
## 5. 特征选择与提取对比

本节对比使用全部特征、特征选择和特征提取三种方案对 SVM 分类性能的影响。

### 5.1 Pipeline 介绍

Pipeline 是 scikit-learn 提供的工具，用于将多个数据处理步骤和模型串联。

```
Pipeline 工作流程
────────────────

原始数据 → [特征选择/提取] → 变换后的数据 → [SVM] → 预测结果
```

| Pipeline 优势 | 说明 |
|---------------|------|
| 防止数据泄露 | 交叉验证时，预处理步骤只在训练集上进行 |
| 代码简洁 | 一行代码完成所有步骤 |
| 易于调参 | 可以对 Pipeline 内所有步骤统一调参 |

### 5.2 性能对比

对比三种方案的交叉验证准确率：

| 方案 | 特征数 | 说明 |
|------|--------|------|
| 全部特征 | 4 | 使用 Iris 全部 4 个特征 |
| Feature Selection | 2 | 使用 ANOVA F-test 选出最佳 2 个特征 |
| PCA | 2 | 使用 PCA 提取 2 个主成分 |

In [ ]:
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import Pipeline

# 使用原始鸢尾花全部4个特征
X_full = iris.data
y = iris.target

print(f"全部特征: {iris.feature_names}")
print(f"样本数: {X_full.shape[0]}, 特征数: {X_full.shape[1]}")

# 特征选择管道
pipe_select = Pipeline([
    ('selector', SelectKBest(f_classif, k=2)),  # 选择最佳2个特征
    ('svm', SVC(kernel='linear', C=1))
])

# 特征提取管道
pipe_extract = Pipeline([
    ('pca', PCA(n_components=2)),  # 降维到2个主成分
    ('svm', SVC(kernel='linear', C=1))
])

print("\nPipeline 创建完成 ✓")

In [ ]:
# 对比性能
results = []
for name, model in [('全部特征 (4D)', SVC(kernel='linear', C=1)),
                      ('Feature Selection (2D)', pipe_select),
                      ('PCA (2D)', pipe_extract)]:
    scores = cross_val_score(model, X_full, y, cv=5)
    results.append({'Method': name, 'Accuracy': f"{scores.mean():.4f}±{scores.std():.4f}"})
    print(f"{name}: {scores.mean():.4f} ± {scores.std():.4f}")

In [ ]:
# 拟合 PCA 管道以获取主成分
pipe_extract.fit(X_full, y)

# 获取 PCA 变换后的数据用于可视化
X_transformed = pipe_extract.named_steps['pca'].transform(X_full)

# 获取拟合后的 SVM 模型
svm_model = pipe_extract.named_steps['svm']

# 可视化决策边界
plt.figure(figsize=(10,6))
plot_decision_regions(X_transformed, y, clf=svm_model, legend=2)
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.title('Decision Boundary after PCA')
plt.show()

---
## 6. 核函数与网格搜索

SVM 的性能很大程度上取决于核函数的选择和超参数的设置。本节使用 GridSearchCV 自动搜索最优参数组合。

### 6.1 GridSearchCV 原理

**网格搜索（Grid Search）**：穷举搜索所有参数组合，找出最优配置。

| 参数网格 | 说明 |
|----------|------|
| kernel | ['linear', 'rbf', 'poly'] — 三种核函数 |
| C | [0.1, 1, 10] — 惩罚系数 |
| gamma | ['scale', 'auto', 0.1, 1] — RBF/Poly 核的参数 |

### 6.2 核函数对比

| 核函数 | 决策边界特点 |
|--------|--------------|
| linear | 直线/平面边界 |
| rbf | 非线性弯曲边界，适应复杂分布 |
| poly | 多项式曲线边界 |

In [ ]:
# 可视化不同核函数效果
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, (ax, kernel) in enumerate(zip(axes, ['linear', 'rbf', 'poly'])):
    model = SVC(kernel=kernel, C=1, gamma='scale')
    model.fit(X_train, y_train)
    plot_decision_regions(X_train, y_train, clf=model, legend=2, ax=ax)
    ax.set_title(f'{kernel.capitalize()} kernel')
    ax.set_xlabel('Sepal Length (cm)')
    ax.set_ylabel('Petal Length (cm)')

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import GridSearchCV

# 定义参数网格
param_grid = {
    'kernel': ['linear', 'rbf', 'poly'],
    'C': [0.1, 1, 10],
    'gamma': ['scale', 'auto', 0.1, 1]
}

# 创建网格搜索对象
grid = GridSearchCV(SVC(), param_grid, cv=5, scoring='accuracy')
grid.fit(X, y)

print(f"最优参数: {grid.best_params_}")
print(f"最优准确率: {grid.best_score_:.4f}")

---
## 7. SVR 回归

支持向量回归 (Support Vector Regression, SVR) 是 SVM 在回归任务上的扩展。

### 7.1 SVR 原理

SVR 的目标是找到一个函数，使预测值与真实值的偏差不超过阈值 $\epsilon$，同时最大化函数的平坦性。

```
SVR 的 epsilon-管道
──────────────────

预测函数 f(x) 在管道中心
      │
      │   ╱─────────────╲   ← 上边界: f(x) + ε
      │  ╱               ╲
      │ ╱    样本点 ○     ╲   ← 管道内的点不计入损失
      │╱                   ╲
      │─────────────────────│   ← 预测函数 f(x)
      │╲                   ╱
      │ ╲    样本点 ○     ╱
      │  ╲               ╱
      │   ╲─────────────╱   ← 下边界: f(x) - ε
```

### 7.2 SVR 关键参数

| 参数 | 含义 |
|------|------|
| C | 惩罚系数，控制管道外样本的影响 |
| epsilon ($\epsilon$) | 管道宽度，管道内不计入损失 |
| kernel | 核函数类型 |
| gamma | RBF 核参数 |

### 7.3 数据加载与探索

使用 California Housing 数据集进行回归实验。该数据集包含加州各区域的房价信息：

| 特征 | 含义 |
|------|------|
| MedInc | 区域人均收入 |
| HouseAge | 房屋平均年龄 |
| AveRooms | 平均房间数 |
| AveBedrms | 平均卧室数 |
| Population | 人口数 |
| AveOccup | 平均入住人数 |
| Latitude | 纬度 |
| Longitude | 经度 |

目标变量：MedHouseVal（区域房屋中位价，单位：10万美元）

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score

# 加载加州房价数据集
housing = fetch_california_housing()
X_housing, y_housing = housing.data, housing.target
feature_names = housing.feature_names

print(f"样本数: {X_housing.shape[0]}, 特征数: {X_housing.shape[1]}")
print(f"特征: {feature_names}")

### 7.4 数据划分

选择人均收入和平均房间数两个特征，便于二维可视化。将数据划分为训练集和测试集。

In [ ]:
# 选择两个特征用于可视化
X_vis = X_housing[:, [0, 2]]  # MedInc 和 AveRooms

# 划分数据集
X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(
    X_vis, y_housing, test_size=0.3, random_state=42
)

print(f"训练集样本数: {len(X_train_h)}")
print(f"测试集样本数: {len(X_test_h)}")

### 7.5 模型训练

创建 SVR 模型并训练：

- `kernel='rbf'`：使用 RBF 核处理非线性关系
- `C=100`：较大的惩罚系数
- `gamma=0.1`：核函数宽度
- `epsilon=0.1`：管道宽度

In [ ]:
# 训练SVR模型
svr_model = SVR(kernel='rbf', C=100, gamma=0.1, epsilon=0.1)
svr_model.fit(X_train_h, y_train_h)

# 预测
y_pred_h = svr_model.predict(X_test_h)

# 评估
mse = mean_squared_error(y_test_h, y_pred_h)
r2 = r2_score(y_test_h, y_pred_h)

print(f"均方误差 (MSE): {mse:.4f}")
print(f"决定系数 (R²): {r2:.4f}")

### 7.6 模型评估

使用以下指标评估回归模型性能：

| 指标 | 含义 | 计算公式 |
|------|------|----------|
| MSE | 均方误差 | $\frac{1}{n}\sum(y_i - \hat{y}_i)^2$ |
| RMSE | 均方根误差 | $\sqrt{\text{MSE}}$ |
| R² | 决定系数 | $1 - \frac{\sum(y_i - \hat{y}_i)^2}{\sum(y_i - \bar{y})^2}$ |

R² 值越接近 1 表示模型拟合越好。

In [ ]:
# 回归预测的可视化结果
# 横轴表示样本真实值，纵轴表示样本预测值
# 对角线是 y=x，越多的点落在对角线上说明预测值和真实值越接近

plt.figure(figsize=(10, 6))
plt.scatter(y_test_h, y_pred_h, alpha=0.5)
plt.plot([y_housing.min(), y_housing.max()], [y_housing.min(), y_housing.max()], 'k--', label='理想预测 (y=x)')
plt.xlabel('True Values')
plt.ylabel('Predictions')
plt.title('SVR Regression Results')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---
## 8. SVR 探索*

本节综合探索特征选择、特征提取、不同核函数和网格搜索对 SVR 回归性能的影响。

### 8.1 探索内容

| 探索方向 | 方法 |
|----------|------|
| 特征选择 | SelectKBest + f_regression |
| 特征提取 | PCA 降维 |
| 参数搜索 | GridSearchCV / RandomizedSearchCV |

In [ ]:
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform

# 特征选择：选择最佳的 2 个特征
selector = SelectKBest(score_func=f_regression, k=2)
X_selected = selector.fit_transform(X_housing, y_housing)
print(f"选择后的特征数: {X_selected.shape[1]}")

In [ ]:
# 特征提取：PCA 降维到 2 个主成分
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_housing)
print(f"PCA 后的特征数: {X_pca.shape[1]}")
print(f"解释方差比: {pca.explained_variance_ratio_}")

In [ ]:
# 网格搜索
param_grid = {
    'C': [1, 10],
    'gamma': ['scale'],
    'epsilon': [0.1, 0.2, 0.5]
}
grid = GridSearchCV(SVR(kernel='rbf'), param_grid, cv=3)
grid.fit(X_selected, y_housing)

print(f"GridSearchCV 最优参数: {grid.best_params_}")
print(f"GridSearchCV 最优 R²: {grid.best_score_:.4f}")

In [ ]:
# 随机搜索
param_distributions = {
    'C': uniform(1, 100),
    'gamma': ['scale', 'auto'],
    'epsilon': uniform(0.01, 0.5)
}

random_search = RandomizedSearchCV(
    SVR(kernel='rbf'),
    param_distributions=param_distributions,
    n_iter=10,
    cv=3,
    scoring='r2',
    random_state=42,
    n_jobs=-1
)
random_search.fit(X_selected, y_housing)

print(f"RandomizedSearchCV 最优参数: {random_search.best_params_}")
print(f"RandomizedSearchCV 最优 R²: {random_search.best_score_:.4f}")

In [ ]:
# 使用最优模型进行预测和可视化
y_pred_best = random_search.predict(X_selected)

plt.figure(figsize=(10, 6))
plt.scatter(y_housing, y_pred_best, alpha=0.5)
plt.xlabel('Actual Price')
plt.ylabel('Predicted Price')
plt.title('SVR Prediction vs Actual (RandomizedSearchCV Best Model)')
plt.plot([y_housing.min(), y_housing.max()], [y_housing.min(), y_housing.max()], 'r--', label='理想预测 (y=x)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---
## 总结

### 方法对比总表

| 方法 | 类型 | 核心思想 | 优点 | 缺点 |
|------|------|----------|------|------|
| SVC | 分类 | 最大化间隔的超平面 | 泛化能力强、核函数灵活 | 大数据集训练慢 |
| SVR | 回归 | epsilon-管道内样本不计损失 | 对异常值鲁棒 | 参数调优复杂 |
| GridSearchCV | 参数搜索 | 穷举所有参数组合 | 保证找到最优解 | 计算量大 |
| RandomizedSearchCV | 参数搜索 | 随机采样参数组合 | 效率高、适合大空间 | 可能错过最优解 |

### 核心知识点

| 知识点 | 要点 |
|--------|------|
| 核函数 | linear 适合线性可分，rbf/poly 适合非线性数据 |
| 超参数 C | 控制间隔宽度 vs 分类错误权衡 |
| 超参数 gamma | RBF 核宽度，越大影响范围越小 |
| Pipeline | 防止数据泄露，串联预处理和模型 |
| 交叉验证 | 更稳健地评估模型泛化能力 |

### 思考题

1. 在什么情况下应该选择 linear 核而不是 rbf 核？
2. GridSearchCV 和 RandomizedSearchCV 各适合什么场景？
3. 特征选择（SelectKBest）和特征提取（PCA）对 SVM 性能的影响有何不同？
4. SVR 的 epsilon 参数如何影响模型的鲁棒性？